# Conversion of CFDBench Cavity/prop dataset to PDEBench INS

Summary:
1) Let channel 3 be geo mask
2) Remove simulations with <20 timesteps
3) For those >20 timesteps, split into segments of 20 and concat accordingly

# Dataset Checks

In [1]:
# Cavity/prop

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/cavity/prop'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0050: u(21, 64, 64), v(21, 64, 64)
case0051: u(6, 64, 64), v(6, 64, 64)
case0052: u(4, 64, 64), v(4, 64, 64)
case0053: u(6, 64, 64), v(6, 64, 64)
case0054: u(6, 64, 64), v(6, 64, 64)
case0055: u(5, 64, 64), v(5, 64, 64)
case0056: u(5, 64, 64), v(5, 64, 64)
case0057: u(32, 64, 64), v(32, 64, 64)
case0058: u(11, 64, 64), v(11, 64, 64)
case0059: u(6, 64, 64), v(6, 64, 64)
case0060: u(5, 64, 64), v(5, 64, 64)
case0061: u(6, 64, 64), v(6, 64, 64)
case0062: u(6, 64, 64), v(6, 64, 64)
case0063: u(5, 64, 64), v(5, 64, 64)
case0064: u(51, 64, 64), v(51, 64, 64)
case0065: u(14, 64, 64), v(14, 64, 64)
case0066: u(9, 64, 64), v(9, 64, 64)
case0067: u(5, 64, 64), v(5, 64, 64)
case0068: u(6, 64, 64), v(6, 64, 64)
case0069: u(6, 64, 64), v(6, 64, 64)
case0070: u(6, 64, 64), v(6, 64, 64)
case0071: u(75, 64, 64), v(75, 64, 64)
case0072: u(18, 64, 64), v(18, 64, 64)
case0073: u(11, 64, 64), v(11, 64, 64)
case0074: u(4, 64, 64), v(4, 64, 64)
case0075: u(5, 64, 64), v(5, 64, 64)
case0076: u(6, 64, 64)

# Padding of Dataset
Based on cavity.py:
```python
def load_case_data(case_dir: Path) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Load from the file that I have preprocessed, and pad the boundary conditions,
    turn into a numpy array of features.

    The shape of both u and v is (time steps, height, width)
    """
    case_params = load_json(case_dir / "case.json")
    # print(case_params)

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)
    # The shape of u and v is (time steps, height, width)

    mask = np.ones_like(u)
    # mask = 1 - mask

    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)
    return features, case_params
```

In [3]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/cavity/prop')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')


100%|██████████| 85/85 [00:01<00:00, 55.90it/s] 

Done


# Sanity Check

In [4]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050: features(21, 3, 64, 64)
case0051: features(6, 3, 64, 64)
case0052: features(4, 3, 64, 64)
case0053: features(6, 3, 64, 64)
case0054: features(6, 3, 64, 64)
case0055: features(5, 3, 64, 64)
case0056: features(5, 3, 64, 64)
case0057: features(32, 3, 64, 64)
case0058: features(11, 3, 64, 64)
case0059: features(6, 3, 64, 64)
case0060: features(5, 3, 64, 64)
case0061: features(6, 3, 64, 64)
case0062: features(6, 3, 64, 64)
case0063: features(5, 3, 64, 64)
case0064: features(51, 3, 64, 64)
case0065: features(14, 3, 64, 64)
case0066: features(9, 3, 64, 64)
case0067: features(5, 3, 64, 64)
case0068: features(6, 3, 64, 64)
case0069: features(6, 3, 64, 64)
case0070: features(6, 3, 64, 64)
case0071: features(75, 3, 64, 64)
case0072: features(18, 3, 64, 64)
case0073: features(11, 3, 64, 64)
case0074: features(4, 3, 64, 64)
case0075: features(5, 3, 64, 64)
case0076: features(6, 3, 64, 64)
case0077: features(6, 3, 64, 64)
case0078: features(1000, 3, 64, 64)
case0079: features(21, 3, 64, 64

# Convergence check
Based on cavity.py/CavityFlowAutoDataset (Lines 298-313):
```python
for i in range(num_steps):
    inp = torch.tensor(inputs[i], dtype=torch.float32)  # (3, h, w)
    out = torch.tensor(outputs[i], dtype=torch.float32)  # (3, h, w)

    # Check for convergence # Computes velocity magnitude field, appends those that did not converge
    inp_magn = torch.sqrt(inp[0] ** 2 + inp[1] ** 2)
    out_magn = torch.sqrt(out[0] ** 2 + out[1] ** 2)
    diff = torch.abs(inp_magn - out_magn).mean()
    if diff < self.stable_state_diff:
        print(f"Converged at {i} out of {num_steps}, {this_case_params}")
        break
    assert not torch.isnan(inp).any()
    assert not torch.isnan(out).any()
    all_inputs.append(inp)  # (3, h, w)
    all_labels.append(out)  # (3, h, w)
    all_case_ids.append(case_id)
```

In [5]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 84/84 [00:01<00:00, 67.61it/s] 

Done


# Sanity Check

In [6]:
# Convergence check

import h5py
import numpy as np
import os

cfd_cavity_prop = '/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_convergence'

for case in sorted(os.listdir(cfd_cavity_prop)):
    case_dir = os.path.join(cfd_cavity_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050: features(21, 3, 64, 64)
case0051: features(6, 3, 64, 64)
case0052: features(4, 3, 64, 64)
case0053: features(6, 3, 64, 64)
case0054: features(6, 3, 64, 64)
case0055: features(5, 3, 64, 64)
case0056: features(5, 3, 64, 64)
case0057: features(32, 3, 64, 64)
case0058: features(11, 3, 64, 64)
case0059: features(6, 3, 64, 64)
case0060: features(5, 3, 64, 64)
case0061: features(6, 3, 64, 64)
case0062: features(6, 3, 64, 64)
case0063: features(5, 3, 64, 64)
case0064: features(51, 3, 64, 64)
case0065: features(14, 3, 64, 64)
case0066: features(9, 3, 64, 64)
case0067: features(5, 3, 64, 64)
case0068: features(6, 3, 64, 64)
case0069: features(6, 3, 64, 64)
case0070: features(6, 3, 64, 64)
case0071: features(75, 3, 64, 64)
case0072: features(18, 3, 64, 64)
case0073: features(11, 3, 64, 64)
case0074: features(4, 3, 64, 64)
case0075: features(5, 3, 64, 64)
case0076: features(6, 3, 64, 64)
case0077: features(6, 3, 64, 64)
case0078: features(1000, 3, 64, 64)
case0079: features(21, 3, 64, 64

# Segmentation
Splits each truncated sequence after convergence step above into fixed length of 20 timesteps, discarding the remainder.

Based on convert_cfdbench.py:

```python
def split_trajectory(data_list, time_step, grid_size=64):
    traj_split = []
    for i, x in enumerate(data_list):
        T = x.shape[0]
        # num_segments = int(np.ceil(T / time_step))
        num_segments = int(np.floor(T / time_step))
        if num_segments < 1:
            continue

        padded_length = num_segments * time_step
        padded_array = x[:padded_length]
        # padded_array = np.zeros((padded_length, *x.shape[1:]))

        # Copy the original data into the padded array
        # padded_array[:T, ...] = x

        # # If needed, pad the last segment with the last frame of the original array
        # if T % time_step != 0:
        #     last_frame = x[-1, ...]
        #     padded_array[T:, ...] = last_frame

        # Reshape the array into segments
        padded_array = F.interpolate(
            torch.from_numpy(padded_array).float(), size=(grid_size, grid_size), mode="bilinear", align_corners=True
        ).numpy()
        padded_array = padded_array.reshape((num_segments, time_step, *padded_array.shape[1:]))

        traj_split.append(padded_array)

    traj_split = np.concatenate(traj_split, axis=0)
    return traj_split
```

In [7]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 84/84 [00:00<00:00, 97.62it/s] 

Done


# Sanity Check

In [8]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050_seg0: features(20, 3, 64, 64)
case0057_seg0: features(20, 3, 64, 64)
case0064_seg0: features(20, 3, 64, 64)
case0064_seg1: features(20, 3, 64, 64)
case0071_seg0: features(20, 3, 64, 64)
case0071_seg1: features(20, 3, 64, 64)
case0071_seg2: features(20, 3, 64, 64)
case0078_seg0: features(20, 3, 64, 64)
case0078_seg1: features(20, 3, 64, 64)
case0078_seg10: features(20, 3, 64, 64)
case0078_seg11: features(20, 3, 64, 64)
case0078_seg12: features(20, 3, 64, 64)
case0078_seg13: features(20, 3, 64, 64)
case0078_seg14: features(20, 3, 64, 64)
case0078_seg15: features(20, 3, 64, 64)
case0078_seg16: features(20, 3, 64, 64)
case0078_seg17: features(20, 3, 64, 64)
case0078_seg18: features(20, 3, 64, 64)
case0078_seg19: features(20, 3, 64, 64)
case0078_seg2: features(20, 3, 64, 64)
case0078_seg20: features(20, 3, 64, 64)
case0078_seg21: features(20, 3, 64, 64)
case0078_seg22: features(20, 3, 64, 64)
case0078_seg23: features(20, 3, 64, 64)
case0078_seg24: features(20, 3, 64, 64)
case0078_s

# Convert to hdf5

In [9]:
import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_final/cavity_prop_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 268/268 [00:27<00:00,  9.67it/s]

Done


# Inference results for converted CAVITY/PROP

In [5]:
#RUN RESULTS FOR CAVITY/PROP

# INFO - 04/06/26 20:34:46 - 0:06:24 - Evaluation Stats (total size = 50)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 50     |    0.107877 |   0.3631 |          0.3103 |          0.3348 |           0.3631 |            0.3357 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.107877 |   0.3631 |          0.3103 |          0.3348 |           0.3631 |            0.3357 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 20:34:46 - 0:06:24 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |     50 | 0.3631 | 0.0284 | 0.3517 | 0.5050 |   0.3523 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 20:34:46 - 0:06:24 - Eval | data loss = 0.107877 | rel l2 = 0.363150 | rel l2 step 1 = 0.310312 | rel l2 step 5 = 0.334834 | rel l2 step 10 = 0.363150 | rel l2 interior = 0.335664
# INFO - 04/06/26 20:34:46 - 0:06:24 -  MEM: 0.00 MB

# Slice out a subset first so inference is faster

import h5py

# input_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_final/cavity_prop_converted.h5'
# output_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cavity_prop/cavity_prop_final/cavity_prop_converted_sliced.h5'

# with h5py.File(input_h5, 'r') as f:
#     with h5py.File(output_h5, 'w') as w:
#         for key in f.keys():
#             subset = f[key][:50]
#
#             w.create_dataset(key, data=subset)
#
# print('Done')

# with h5py.File(output_h5, 'r') as f:
#     print(f.keys())
#     print(f['velocity'])
#     print(f['particles'])

<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (50, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (50, 20, 512, 512, 1), type "<f4">


# Process is repeated for Cavity/BC -> PDEBench INS

In [10]:
# Cavity/bc
# Let channel 3 be geo mask
# Remove simulations with <20 timesteps
# For those >20 timesteps, split into segments of 20 and concat accordingly

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/cavity/bc'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0000: u(10, 64, 64), v(10, 64, 64)
case0001: u(11, 64, 64), v(11, 64, 64)
case0002: u(13, 64, 64), v(13, 64, 64)
case0003: u(14, 64, 64), v(14, 64, 64)
case0004: u(15, 64, 64), v(15, 64, 64)
case0005: u(16, 64, 64), v(16, 64, 64)
case0006: u(17, 64, 64), v(17, 64, 64)
case0007: u(18, 64, 64), v(18, 64, 64)
case0008: u(20, 64, 64), v(20, 64, 64)
case0009: u(21, 64, 64), v(21, 64, 64)
case0010: u(21, 64, 64), v(21, 64, 64)
case0011: u(22, 64, 64), v(22, 64, 64)
case0012: u(23, 64, 64), v(23, 64, 64)
case0013: u(24, 64, 64), v(24, 64, 64)
case0014: u(25, 64, 64), v(25, 64, 64)
case0015: u(26, 64, 64), v(26, 64, 64)
case0016: u(27, 64, 64), v(27, 64, 64)
case0017: u(28, 64, 64), v(28, 64, 64)
case0018: u(28, 64, 64), v(28, 64, 64)
case0019: u(29, 64, 64), v(29, 64, 64)
case0020: u(30, 64, 64), v(30, 64, 64)
case0021: u(31, 64, 64), v(31, 64, 64)
case0022: u(32, 64, 64), v(32, 64, 64)
case0023: u(33, 64, 64), v(33, 64, 64)
case0024: u(36, 64, 64), v(36, 64, 64)
case0025: u(38, 64, 64), 

In [11]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/cavity/bc')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')


100%|██████████| 51/51 [00:01<00:00, 40.15it/s] 

Done


In [12]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0000: features(10, 3, 64, 64)
case0001: features(11, 3, 64, 64)
case0002: features(13, 3, 64, 64)
case0003: features(14, 3, 64, 64)
case0004: features(15, 3, 64, 64)
case0005: features(16, 3, 64, 64)
case0006: features(17, 3, 64, 64)
case0007: features(18, 3, 64, 64)
case0008: features(20, 3, 64, 64)
case0009: features(21, 3, 64, 64)
case0010: features(21, 3, 64, 64)
case0011: features(22, 3, 64, 64)
case0012: features(23, 3, 64, 64)
case0013: features(24, 3, 64, 64)
case0014: features(25, 3, 64, 64)
case0015: features(26, 3, 64, 64)
case0016: features(27, 3, 64, 64)
case0017: features(28, 3, 64, 64)
case0018: features(28, 3, 64, 64)
case0019: features(29, 3, 64, 64)
case0020: features(30, 3, 64, 64)
case0021: features(31, 3, 64, 64)
case0022: features(32, 3, 64, 64)
case0023: features(33, 3, 64, 64)
case0024: features(36, 3, 64, 64)
case0025: features(38, 3, 64, 64)
case0026: features(40, 3, 64, 64)
case0027: features(44, 3, 64, 64)
case0028: features(43, 3, 64, 64)
case0029: feat

In [13]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 50/50 [00:01<00:00, 47.27it/s] 

Done


In [14]:
# Convergence check

import h5py
import numpy as np
import os

cfd_cavity_prop = '/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_convergence'

for case in sorted(os.listdir(cfd_cavity_prop)):
    case_dir = os.path.join(cfd_cavity_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0000: features(8, 3, 64, 64)
case0001: features(11, 3, 64, 64)
case0002: features(13, 3, 64, 64)
case0003: features(14, 3, 64, 64)
case0004: features(15, 3, 64, 64)
case0005: features(16, 3, 64, 64)
case0006: features(17, 3, 64, 64)
case0007: features(18, 3, 64, 64)
case0008: features(20, 3, 64, 64)
case0009: features(21, 3, 64, 64)
case0010: features(21, 3, 64, 64)
case0011: features(22, 3, 64, 64)
case0012: features(23, 3, 64, 64)
case0013: features(24, 3, 64, 64)
case0014: features(25, 3, 64, 64)
case0015: features(26, 3, 64, 64)
case0016: features(27, 3, 64, 64)
case0017: features(28, 3, 64, 64)
case0018: features(28, 3, 64, 64)
case0019: features(29, 3, 64, 64)
case0020: features(30, 3, 64, 64)
case0021: features(31, 3, 64, 64)
case0022: features(32, 3, 64, 64)
case0023: features(33, 3, 64, 64)
case0024: features(36, 3, 64, 64)
case0025: features(38, 3, 64, 64)
case0026: features(40, 3, 64, 64)
case0027: features(44, 3, 64, 64)
case0028: features(43, 3, 64, 64)
case0029: featu

In [15]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 50/50 [00:00<00:00, 61.78it/s] 

Done


In [16]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0008_seg0: features(20, 3, 64, 64)
case0009_seg0: features(20, 3, 64, 64)
case0010_seg0: features(20, 3, 64, 64)
case0011_seg0: features(20, 3, 64, 64)
case0012_seg0: features(20, 3, 64, 64)
case0013_seg0: features(20, 3, 64, 64)
case0014_seg0: features(20, 3, 64, 64)
case0015_seg0: features(20, 3, 64, 64)
case0016_seg0: features(20, 3, 64, 64)
case0017_seg0: features(20, 3, 64, 64)
case0018_seg0: features(20, 3, 64, 64)
case0019_seg0: features(20, 3, 64, 64)
case0020_seg0: features(20, 3, 64, 64)
case0021_seg0: features(20, 3, 64, 64)
case0022_seg0: features(20, 3, 64, 64)
case0023_seg0: features(20, 3, 64, 64)
case0024_seg0: features(20, 3, 64, 64)
case0025_seg0: features(20, 3, 64, 64)
case0026_seg0: features(20, 3, 64, 64)
case0026_seg1: features(20, 3, 64, 64)
case0027_seg0: features(20, 3, 64, 64)
case0027_seg1: features(20, 3, 64, 64)
case0028_seg0: features(20, 3, 64, 64)
case0028_seg1: features(20, 3, 64, 64)
case0029_seg0: features(20, 3, 64, 64)
case0029_seg1: features(2

In [17]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_final/cavity_bc_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 232/232 [00:24<00:00,  9.49it/s]

Done


# Inference results for converted CAVITY/BC

In [2]:
# CAVITY/BC RESULTS

# INFO - 04/06/26 20:20:55 - 0:06:29 - Evaluation Stats (total size = 50)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 50     |    0.141194 |   0.4057 |          0.3549 |          0.3789 |           0.4057 |            0.3785 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.141194 |   0.4057 |          0.3549 |          0.3789 |           0.4057 |            0.3785 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 20:20:55 - 0:06:29 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |     50 | 0.4057 | 0.0230 | 0.3727 | 0.4456 |   0.4059 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 20:20:55 - 0:06:29 - Eval | data loss = 0.141194 | rel l2 = 0.405691 | rel l2 step 1 = 0.354905 | rel l2 step 5 = 0.378869 | rel l2 step 10 = 0.405691 | rel l2 interior = 0.378500
# INFO - 04/06/26 20:20:55 - 0:06:29 -  MEM: 0.00 MB

# Slice out a subset first so inference is faster

# import h5py
#
# input_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_final/cavity_bc_converted.h5'
# output_h5 = '/Volumes/T7/CFDBench/Processed_experiment/cavity_bc/cavity_bc_final/cavity_bc_converted_sliced.h5'
#
# with h5py.File(input_h5, 'r') as f:
#     with h5py.File(output_h5, 'w') as w:
#         for key in f.keys():
#             subset = f[key][:50]
#
#             w.create_dataset(key, data=subset)
#
# print('Done')

# with h5py.File(output_h5, 'r') as f:
#     print(f.keys())
#     print(f['velocity'])
#     print(f['particles'])

<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (50, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (50, 20, 512, 512, 1), type "<f4">


# Process is repeated for CAVITY/GEO

In [18]:
# Cavity/geo
# Let channel 3 be geo mask
# Remove simulations with <20 timesteps
# For those >20 timesteps, split into segments of 20 and concat accordingly

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/cavity/geo'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0001: u(23, 64, 64), v(23, 64, 64)
case0002: u(20, 64, 64), v(20, 64, 64)
case0003: u(20, 64, 64), v(20, 64, 64)
case0004: u(21, 64, 64), v(21, 64, 64)
case0005: u(37, 64, 64), v(37, 64, 64)
case0006: u(43, 64, 64), v(43, 64, 64)
case0007: u(43, 64, 64), v(43, 64, 64)
case0008: u(42, 64, 64), v(42, 64, 64)
case0009: u(60, 64, 64), v(60, 64, 64)
case0010: u(203, 64, 64), v(203, 64, 64)
case0011: u(68, 64, 64), v(68, 64, 64)
case0012: u(84, 64, 64), v(84, 64, 64)
case0013: u(137, 64, 64), v(137, 64, 64)
case0014: u(113, 64, 64), v(113, 64, 64)
case0015: u(23, 64, 64), v(23, 64, 64)
case0016: u(100, 64, 64), v(100, 64, 64)
case0017: u(87, 64, 64), v(87, 64, 64)
case0018: u(37, 64, 64), v(37, 64, 64)
case0019: u(45, 64, 64), v(45, 64, 64)
case0020: u(44, 64, 64), v(44, 64, 64)
case0021: u(100, 64, 64), v(100, 64, 64)
case0022: u(74, 64, 64), v(74, 64, 64)
case0023: u(72, 64, 64), v(72, 64, 64)
case0024: u(194, 64, 64), v(194, 64, 64)


In [19]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/cavity/geo')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')


100%|██████████| 25/25 [00:00<00:00, 57.10it/s]

Done


In [20]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 24/24 [00:00<00:00, 149.62it/s]

Done


In [21]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 24/24 [00:00<00:00, 65.03it/s]

Done


In [22]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0001_seg0: features(20, 3, 64, 64)
case0002_seg0: features(20, 3, 64, 64)
case0003_seg0: features(20, 3, 64, 64)
case0004_seg0: features(20, 3, 64, 64)
case0005_seg0: features(20, 3, 64, 64)
case0006_seg0: features(20, 3, 64, 64)
case0006_seg1: features(20, 3, 64, 64)
case0007_seg0: features(20, 3, 64, 64)
case0007_seg1: features(20, 3, 64, 64)
case0008_seg0: features(20, 3, 64, 64)
case0008_seg1: features(20, 3, 64, 64)
case0009_seg0: features(20, 3, 64, 64)
case0009_seg1: features(20, 3, 64, 64)
case0009_seg2: features(20, 3, 64, 64)
case0010_seg0: features(20, 3, 64, 64)
case0010_seg1: features(20, 3, 64, 64)
case0010_seg2: features(20, 3, 64, 64)
case0010_seg3: features(20, 3, 64, 64)
case0010_seg4: features(20, 3, 64, 64)
case0010_seg5: features(20, 3, 64, 64)
case0010_seg6: features(20, 3, 64, 64)
case0010_seg7: features(20, 3, 64, 64)
case0010_seg8: features(20, 3, 64, 64)
case0010_seg9: features(20, 3, 64, 64)
case0011_seg0: features(20, 3, 64, 64)
case0011_seg1: features(2

In [23]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/cavity_geo/cavity_geo_final/cavity_geo_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 77/77 [00:07<00:00,  9.65it/s]

Done


# Inference Results for converted CAVITY/GEO

In [ ]:
# CAVITY/GEO RESULTS
# INFO - 04/06/26 09:28:05 - 0:11:09 - Evaluation Stats (total size = 77)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 77     |    0.201489 |   0.4493 |          0.3976 |          0.4235 |           0.4493 |            0.4207 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.201489 |   0.4493 |          0.3976 |          0.4235 |           0.4493 |            0.4207 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 09:28:05 - 0:11:09 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |     77 | 0.4493 | 0.0592 | 0.3563 | 0.6211 |   0.4415 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 09:28:05 - 0:11:09 - Eval | data loss = 0.201489 | rel l2 = 0.449329 | rel l2 step 1 = 0.397646 | rel l2 step 5 = 0.423526 | rel l2 step 10 = 0.449329 | rel l2 interior = 0.420704
# INFO - 04/06/26 09:28:05 - 0:11:09 -  MEM: 0.00 MB


# Inverted Masking results
I inverted the values of the mask, as per the code block in cavity.py below:

```python
def load_case_data(case_dir: Path) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Load from the file that I have preprocessed, and pad the boundary conditions,
    turn into a numpy array of features.

    The shape of both u and v is (time steps, height, width)
    """
    case_params = load_json(case_dir / "case.json")
    # print(case_params)

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)
    # The shape of u and v is (time steps, height, width)

    # mask = np.ones_like(u)
    mask = 1 - mask #<--- This was used instead of the line above, so mask is now 0, instead of 1

    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)
    return features, case_params

```

In [ ]:
# CFDBENCH INVERTED
# INFO - 04/08/26 11:07:04 - 0:10:14 - Evaluation Stats (total size = 82)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | cfdbench     | 2     | 82     |    0.393060 |   0.2169 |          0.2079 |          0.2112 |           0.2169 |            0.1960 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.393060 |   0.2169 |          0.2079 |          0.2112 |           0.2169 |            0.1960 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/08/26 11:07:04 - 0:10:14 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | cfdbench |     82 | 0.2169 | 0.0246 | 0.1928 | 0.4066 |   0.2164 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/08/26 11:07:04 - 0:10:14 - Eval | data loss = 0.393060 | rel l2 = 0.216937 | rel l2 step 1 = 0.207917 | rel l2 step 5 = 0.211208 | rel l2 step 10 = 0.216937 | rel l2 interior = 0.196006
# INFO - 04/08/26 11:07:04 - 0:10:14 -  MEM: 0.00 MB

In [ ]:
#CFDBENCH ORIGINAL
# INFO - 04/08/26 10:45:06 - 0:10:16 - Evaluation Stats (total size = 82)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | cfdbench     | 2     | 82     |    0.000075 |   0.0032 |          0.0034 |          0.0032 |           0.0032 |            0.0030 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.000075 |   0.0032 |          0.0034 |          0.0032 |           0.0032 |            0.0030 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/08/26 10:45:06 - 0:10:16 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | cfdbench |     82 | 0.0032 | 0.0016 | 0.0022 | 0.0141 |   0.0031 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/08/26 10:45:06 - 0:10:16 - Eval | data loss = 0.000075 | rel l2 = 0.003210 | rel l2 step 1 = 0.003404 | rel l2 step 5 = 0.003180 | rel l2 step 10 = 0.003210 | rel l2 interior = 0.003020
# INFO - 04/08/26 10:45:06 - 0:10:16 -  MEM: 0.00 MB

# Conclusion
Inverted mask leads to horrible results compared to the original.